# Learning LLM Engineering: Mastering Tokenizers

Tokenizers are the first and last step in every LLM pipeline. They convert human-readable text into numerical Token IDs that models can process, and vice-versa.

In [ ]:
# Step 1: Initialize the environment with specific versions to ensure reproducibility of my experiments.
!pip install -q --upgrade datasets==3.6.0 transformers==4.57.6

In [ ]:
# Importing core utilities for model access and tokenization.
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer

In [ ]:
# Logging into HF and verifying GPU status to ensure I have the hardware for faster inference tests.
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

gpu_info = !nvidia-smi
if 'failed' not in '\n'.join(gpu_info):
    print("GPU is active. Ready for tokenization benchmarks.")

In [ ]:
# Loading the Llama 3.1 base tokenizer to analyze its specific sub-word splitting logic.
tokenizer = AutoTokenizer.from_pretrained('meta-llama/Meta-Llama-3.1-8B', trust_remote_code=True)

In [ ]:
# Testing the encoding process: converting raw strings into the integer IDs the model consumes.
text = "I am excited to show Tokenizers in action to my LLM engineers"
tokens = tokenizer.encode(text)
display(tokens)

In [ ]:
# Benchmarking token efficiency. This ratio helps me calculate context window usage and potential API costs.
print(f"Chars: {len(text)} | Words: {len(text.split())} | Tokens: {len(tokens)}")

In [ ]:
# Verifying the decoding loop to ensure the original text is reconstructed accurately from the IDs.
tokenizer.decode(tokens)

In [ ]:
# Visualizing how the tokenizer fragments specific words into individual tokens.
tokenizer.batch_decode(tokens)

In [ ]:
# Identifying reserved and special tokens (like <|begin_of_text|>) that dictate the model's control flow.
tokenizer.get_added_vocab()

In [ ]:
# Checking total vocabulary capacity; larger vocabs usually mean better compression for diverse languages/code.
len(tokenizer.vocab)

## My notes on Chat Templates

I've learned that 'Instruct' models need specific formatting to distinguish between system instructions and user input. `apply_chat_template` is the tool I'll use to automate this formatting so I don't have to manually write complex tags.

In [ ]:
# Switching to the 'Instruct' variant to see how formatting tags change for chat-based interactions.
tokenizer = AutoTokenizer.from_pretrained('meta-llama/Meta-Llama-3.1-8B-Instruct', trust_remote_code=True)

In [ ]:
# Using apply_chat_template to handle the complex Llama 3 header tags automatically.
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Tell a light-hearted joke for a room of Data Scientists"}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(prompt)

## The Core Realization

I used to think LLMs processed Python lists or dictionaries directly. Now I see the pipeline clearly:
1. My message list is turned into a single formatted string.
2. That string is broken into sub-word tokens.
3. Tokens are mapped to IDs.

**The input is always just a sequence of numbers.**

## Comparative Analysis of Tokenizers

Different families (Microsoft, DeepSeek, Qwen) use different vocabularies and splitting logic. As an engineer, choosing the right tokenizer affects latency and cost.

In [ ]:
# Defining model constants for a comparative study across different architecture families.
PHI4 = "microsoft/Phi-4-mini-instruct"
DEEPSEEK = "deepseek-ai/DeepSeek-V3.1"
QWEN_CODER = "Qwen/Qwen2.5-Coder-7B-Instruct"

In [ ]:
# Direct comparison: observing how Llama and Phi-4 map the same semantic string to different ID spaces.
phi4_tokenizer = AutoTokenizer.from_pretrained(PHI4)

print("Llama IDs:", tokenizer.encode(text))
print("Phi-4 IDs:", phi4_tokenizer.encode(text))

In [ ]:
# Comparing Chat Template syntax: seeing how each model separates system and user turns.
print("Llama Template Logic:")
print(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
print("\nPhi-4 Template Logic:")
print(phi4_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

In [ ]:
# Extending my analysis to DeepSeek to compare its vocabulary efficiency against Llama and Phi.
deepseek_tokenizer = AutoTokenizer.from_pretrained(DEEPSEEK)

print("Llama IDs:", tokenizer.encode(text))
print("Phi-4 IDs:", phi4_tokenizer.encode(text))
print("DeepSeek IDs:", deepseek_tokenizer.encode(text))

In [ ]:
print("Llama:")
print(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
print("\nPhi:")
print(phi4_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
print("\nDeepSeek:")
print(deepseek_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

In [ ]:
# Testing Qwen Coder: checking if it preserves whitespace and syntax keywords differently for programming tasks.
qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_CODER)
code = """
def hello_world(person):
  print("Hello", person)
"""
tokens = qwen_tokenizer.encode(code)
for t in tokens:
    print(f"{t} -> '{qwen_tokenizer.decode(t)}'")